# w9_docview_5fold.ipynb — 文档视图消融五折 @512(Table A3 折级版)

User (2026-07-20): 固定分割上 present/.657 vs wllm/.608 vs nodoc/.603 的差
全部落在 ±0.03-0.04 选点摆动带内(轨迹峰 .672/.667/.603)——单分割不可读,
只有五折能定谳。本 notebook = {nodoc, wllm} × 5 folds @512/2000ep,
seed=fold(与 wave-1 i2ce@512 折配对,可做逐折配对差)。present 参照行 =
wave-1 `wcle_i2ce_icetf` @512 五折(read-only,桶上已有)。CV worker 已
移植 `--wiki-src llm`(后缀 _wllm);nodoc 臂原生支持。ZS-only,rvsel。
AUTO-STOPS。


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_cv_out"       # CV campaign out dir

# (recipe, extra worker args, name suffix)
JOBS = [("wcle_nodoc_i2ce_icetf", [], ""),
        ("wcle_i2ce_icetf", ["--wiki-src", "llm"], "_wllm")]
REF = "wcle_i2ce_icetf"                # wave-1 @512 folds (read-only)
CAPS = [512]
N_FOLDS = 5
EPOCHS, CKPT_EVERY, CKPT_SEEDS, TOPUP_SEEDS = 2000, 50, 2, 10   # ZS: seeds unused

SAFETY = 0.85
RESERVE_GIB = 1.5

def nm_of(r, k, cap, sfx=""):
    return (f"w9cv_{r}_fold{k}" + (f"_g{cap}" if cap != 512 else "") + sfx)

os.makedirs(OUT_DIR, exist_ok=True)
print("jobs:", len(JOBS) * N_FOLDS, f"({len(JOBS)} arms x {N_FOLDS} folds) @512/{EPOCHS}ep")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (same file set as w9_a100.ipynb).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "wiki_llm_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# VRAM-BUDGET SCHEDULER over the (job x fold) grid @512.
import os, subprocess, tempfile, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
MEAS_OUT = os.path.join(tempfile.gettempdir(), "w9_measure_out")
os.makedirs(MEAS_OUT, exist_ok=True)
gpus = J.detect_gpus()

def _smi_mib(field, g):
    out = subprocess.check_output(
        ["nvidia-smi", f"--query-gpu={field}", "--format=csv,noheader,nounits",
         "-i", str(g)]).decode().strip().split("\n")[0]
    return int(out) * 2**20

free = {g: _smi_mib("memory.free", g) for g in gpus}
budget = {g: int(free[g] * SAFETY - RESERVE_GIB * 2**30) for g in gpus}
print(f"[vram] budgets {[f'{budget[g] / 2**30:.0f}G' for g in gpus]}")

todo0 = []
for cap in CAPS:
    for r, extra, sfx in JOBS:
        for k in range(N_FOLDS):
            nm = nm_of(r, k, cap, sfx)
            if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
                print(f"[skip] {nm} at {EPOCHS}"); continue
            todo0.append((r, tuple(extra), sfx, k, cap, nm))

cost = {}
for r, extra, sfx in JOBS:
    key = r + sfx
    if not any(t[0] == r and t[2] == sfx for t in todo0):
        continue
    tf = Path(tempfile.gettempdir()) / f"w9cv_vram_{key}.txt"
    tf.unlink(missing_ok=True)
    cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           MEAS_OUT, "--repo", REPO, "--arm", r, "--fold", "0",
           "--n-folds", str(N_FOLDS), "--anchor-cap", str(CAPS[0]),
           "--epochs", "1", "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--measure-vram", str(tf)] + list(extra)
    print(f"[warmup] {key} ...", flush=True)
    with open(logd / f"measure_cv_{key}.log", "w") as fh:
        subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0]))
    cost[key] = int(tf.read_text()) if tf.exists() else budget[gpus[0]] + 1
    print(f"[warmup] {key}: {cost[key] / 2**30:.2f}G", flush=True)

todo = sorted(((r, e, sfx, k, cap, nm, cost[r + sfx])
               for r, e, sfx, k, cap, nm in todo0), key=lambda x: -x[6])
now_used = {g: 0 for g in gpus}
now_cnt = {g: 0 for g in gpus}
MAX_CO = 4        # host-RAM guard: ~8.5G boot transient per worker
LAUNCH_STAGGER = 60
fails = []
cvn = threading.Condition()

def run_job(g, r, extra, sfx, k, cap, nm, c):
    try:
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
        cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
               OUT_DIR, "--repo", REPO, "--arm", r, "--fold", str(k),
               "--n-folds", str(N_FOLDS), "--anchor-cap", str(cap),
               "--epochs", str(EPOCHS), "--ckpt-every", str(CKPT_EVERY),
               "--ckpt-seeds", str(CKPT_SEEDS), "--topup-seeds", str(TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")] + list(extra)
        t0 = time.time()
        with open(logd / f"{nm}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
        if p.returncode != 0:
            (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
        print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
              f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)
    finally:
        with cvn:
            now_used[g] -= c
            now_cnt[g] -= 1
            cvn.notify_all()

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
active = []
with cvn:
    pending = list(todo)
    while pending or any(t.is_alive() for t in active):
        prog = False
        i = 0
        while i < len(pending):
            r, e, sfx, k, cap, nm, c = pending[i]
            fit = [g for g in gpus if now_cnt[g] < MAX_CO
                   and (now_used[g] + c <= budget[g] or now_used[g] == 0)]
            if not fit:
                i += 1; continue
            g = min(fit, key=lambda g: now_used[g])
            now_used[g] += c
            now_cnt[g] += 1
            th = threading.Thread(target=run_job, args=(g, r, e, sfx, k, cap, nm, c),
                                  daemon=True)
            active.append(th); th.start(); pending.pop(i)
            print(f"[sched] {nm} -> gpu{g} ({c / 2**30:.1f}G, used "
                  f"{now_used[g] / 2**30:.1f}/{budget[g] / 2**30:.0f}G)", flush=True)
            prog = True
            cvn.wait(timeout=LAUNCH_STAGGER)
        active = [t for t in active if t.is_alive()]
        if not prog:
            cvn.wait(timeout=3)
stop_evt.set()
for t in active:
    t.join()
print(f"FINAL grid drained; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)


In [ ]:
# THE TABLE-A3 FOLD VERSION: present (wave-1 ref) / nodoc / wllm,
# 5-fold mean+-std + paired per-fold deltas vs present.
import json
import numpy as np
from pathlib import Path

def _load(nm):
    p = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    return json.loads(p.read_text()) if p.exists() else None

ROWS = [("present (wave-1 i2ce)", REF, ""),
        ("no document view", "wcle_nodoc_i2ce_icetf", ""),
        ("LLM-rewritten docs", "wcle_i2ce_icetf", "_wllm")]
KEYS = ["nm_neutral", "h5_neutral", "nm_noname", "h5_noname",
        "tag_neutral", "tag_noname"]
per = {}
for lab, r, sfx in ROWS:
    ds = [_load(nm_of(r, k, 512, sfx)) for k in range(N_FOLDS)]
    per[lab] = ds
    got = [d for d in ds if d]
    line = f"{lab:24s} [{len(got)}/5]"
    if got:
        for key in KEYS:
            v = np.array([d[key] for d in got])
            line += f"  {key} {v.mean():.3f}+-{v.std():.3f}"
    print(line)
ref = per["present (wave-1 i2ce)"]
for lab in ("no document view", "LLM-rewritten docs"):
    ds = per[lab]
    pairs = [(d["nm_noname"] - rf["nm_noname"])
             for d, rf in zip(ds, ref) if d and rf]
    if pairs:
        print(f"paired non delta {lab}: {np.mean(pairs):+.3f} "
              f"(wins {sum(p > 0 for p in pairs)}/{len(pairs)})")


In [ ]:
# AUTO-STOP: stop THIS pod when the queue has finished (results live on the
# network volume; idle GPU time is pure waste). Uses the hardened ladder in
# VICReg_review/pod_selfstop.py. Set AUTO_STOP=False to keep the pod alive.
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    if fails:
        print(f"NOTE: {len(fails)} job(s) FAILED -- logs in {OUT_DIR}/logs; "
              "stopping anyway to avoid idle burn.")
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- remember to stop the pod yourself.")